# optimizer-init-params-list — worked example 2: Multi-step SGD correctness via list-materialized params

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-init-params-list`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Materializing parameters into a list at `__init__` time is what allows an optimizer to run correctly over many training iterations. If the optimizer receives `model.parameters()` (a generator), the list call consumes it once and stores references to the actual parameter tensors — not copies. Subsequent steps still see the live tensors with their updated `.data` and freshly computed `.grad`.

## Worked solution

**Step 1 — Build the optimizer with `list(params)`.**
In `__init__`, we call `self.params = list(params)`. The list contains references to the actual parameter tensor objects. When `model.parameters()` is the input, each parameter tensor is referenced exactly once.

**Step 2 — Run multiple steps.**
Each call to `.step()` iterates `self.params`. Because it is a list, re-iteration works every time. The parameters are updated in place via `p.data -= lr * p.grad`.

**Step 3 — Compare against PyTorch's SGD.**
We run both our `SimpleSGD` and `torch.optim.SGD` from identical initial weights and gradients for 5 steps. The final parameter values must match within floating-point tolerance.

**Step 4 — Confirm list length.**
We also verify that `len(self.params)` equals the number of parameters in the model — the list captured all of them.

In [ ]:
import torch as t
import torch.nn as nn

class SimpleSGD:
    def __init__(self, params, lr):
        self.params = list(params)   # critical
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- exercise it ---
t.manual_seed(0)
model_a = nn.Sequential(nn.Linear(4, 3), nn.Linear(3, 2))
model_b = nn.Sequential(nn.Linear(4, 3), nn.Linear(3, 2))

# Copy weights so both models start identical
for pa, pb in zip(model_a.parameters(), model_b.parameters()):
    pb.data.copy_(pa.data)

opt_ours  = SimpleSGD(model_a.parameters(), lr=0.05)
opt_torch = t.optim.SGD(model_b.parameters(), lr=0.05)

data_x = t.randn(6, 4, generator=t.Generator().manual_seed(7))

for step in range(5):
    # Our optimizer
    out_a = model_a(data_x).sum()
    out_a.backward()
    opt_ours.step()
    opt_ours.zero_grad()
    # PyTorch optimizer
    out_b = model_b(data_x).sum()
    out_b.backward()
    opt_torch.step()
    opt_torch.zero_grad()

for pa, pb in zip(model_a.parameters(), model_b.parameters()):
    assert t.allclose(pa, pb, atol=1e-5), f'Mismatch: {pa} vs {pb}'

print(f'param count captured: {len(opt_ours.params)}')
print('All 5-step params match PyTorch SGD.')